# Stablecoin Bad-Data Diagnostic

For each suspicious row (High or Low outside the ±$0.14 peg band), this notebook:

1. **Fetches the Binance.US trade-level bulk file** for that exact date — these are the raw executed transactions, one row per trade. The real daily High and Low are computed directly from trade prices, bypassing the kline aggregation that produced the bad values.
2. **Cross-checks with Kraken and Coinbase** for the same date as independent US-exchange confirmation.
3. **Produces a verdict table** — for each suspicious row: what did actual trades say, what did cross-exchange sources say, and what the corrected OHLCV values should be.

## 1. Suspicious Rows to Investigate

In [1]:
DIAGNOSTIC_OUTPUT_CSV = "stablecoin_diagnostic.csv"
INPUT_EXCEL_FILE       = "ohlcv_final.xlsx"

import requests
import zipfile
import hashlib
import io
import time
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta

SUSPICIOUS = [
    ("USDT/USD", "2020-02-19", 5.00, 1.00),
    ("USDT/USD", "2020-03-13", 4.00, 1.00),
    ("USDT/USD", "2023-05-05", 3.48, 1.00),
    ("USDT/USD", "2023-06-06", 4.50, 1.00),
    ("USDT/USD", "2023-06-07", 1.70, 1.00),
    ("USDT/USD", "2023-07-14", 0.92, 0.70),
    ("USDC/USD", "2021-01-12", 1.50, 1.00),
    ("USDC/USD", "2021-01-13", 2.00, 1.00),
    ("USDC/USD", "2021-05-19", 1.35, 0.98),
    ("USDC/USD", "2022-12-13", 4.00, 1.00),
]

BINANCE_BULK_SYMBOLS = {"USDT/USD": "USDTUSD", "USDC/USD": "USDCUSD"}
KRAKEN_SYMBOLS       = {"USDT/USD": "USDTUSD", "USDC/USD": "USDCUSD"}
COINBASE_SYMBOLS     = {"USDT/USD": "USDT-USD"}

DAILY_TRADE_BASE = "https://data.binance.us/public_data/spot/daily/trades"
DAILY_KLINE_BASE = "https://data.binance.us/public_data/spot/daily/klines"

PEG_HIGH = 1.14
PEG_LOW  = 0.86

print(f"Suspicious rows to investigate: {len(SUSPICIOUS)}")
print(f"  USDT/USD: {sum(1 for r in SUSPICIOUS if r[0]=='USDT/USD')}")
print(f"  USDC/USD: {sum(1 for r in SUSPICIOUS if r[0]=='USDC/USD')}")

Suspicious rows to investigate: 10
  USDT/USD: 6
  USDC/USD: 4


## 2. Fetch Functions

In [2]:
def _download(url):
    """Download a URL. Returns (bytes, status_code)."""
    try:
        r = requests.get(url, timeout=60)
        return r.content, r.status_code
    except Exception as e:
        return None, str(e)


def _verify_checksum(data_bytes, checksum_url):
    try:
        r = requests.get(checksum_url, timeout=15)
        if r.status_code != 200:
            return True
        expected = r.text.strip().split()[0].lower()
        return hashlib.sha256(data_bytes).hexdigest().lower() == expected
    except Exception:
        return True


def get_binance_trade_range(pair, date_str):
    """
    Download the raw trade file for a specific date from Binance.US bulk.
    Trade file format: id, price, qty, quote_qty, time, is_buyer_maker
    Returns dict with real min/max/open/close from actual executed trades,
    or None if file not available.
    """
    symbol = BINANCE_BULK_SYMBOLS[pair]
    url    = f"{DAILY_TRADE_BASE}/{symbol}/{symbol}-trades-{date_str}.zip"
    cksum  = url + ".CHECKSUM"

    data, status = _download(url)
    if status == 404 or data is None:
        return None

    if not _verify_checksum(data, cksum):
        return {"error": "checksum mismatch"}

    prices = []
    try:
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            csv_name = [n for n in zf.namelist() if n.endswith(".csv")][0]
            with zf.open(csv_name) as f:
                for line in f:
                    parts = line.decode().strip().split(",")
                    if not parts or not parts[0].isdigit():
                        continue
                    prices.append(float(parts[1]))
    except Exception as e:
        return {"error": str(e)}

    if not prices:
        return None

    return {
        "n_trades":   len(prices),
        "trade_open": prices[0],
        "trade_high": max(prices),
        "trade_low":  min(prices),
        "trade_close":prices[-1],
    }


def get_binance_kline(pair, date_str):
    """Re-fetch the daily kline file for a date to confirm reported OHLCV."""
    symbol = BINANCE_BULK_SYMBOLS[pair]
    url    = f"{DAILY_KLINE_BASE}/{symbol}/1d/{symbol}-1d-{date_str}.zip"
    data, status = _download(url)
    if status == 404 or data is None:
        return None
    try:
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            csv_name = [n for n in zf.namelist() if n.endswith(".csv")][0]
            with zf.open(csv_name) as f:
                for line in f:
                    parts = line.decode().strip().split(",")
                    if not parts or not parts[0].isdigit():
                        continue
                    return {
                        "kline_open":  float(parts[1]),
                        "kline_high":  float(parts[2]),
                        "kline_low":   float(parts[3]),
                        "kline_close": float(parts[4]),
                    }
    except Exception as e:
        return {"error": str(e)}
    return None


def get_kraken_ohlcv(pair, date_str):
    """Fetch Kraken daily OHLCV for a specific date."""
    if pair not in KRAKEN_SYMBOLS:
        return None
    symbol   = KRAKEN_SYMBOLS[pair]
    url      = "https://api.kraken.com/0/public/OHLC"
    date_dt  = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    since    = int((date_dt - timedelta(days=1)).timestamp())
    end_ts   = int((date_dt + timedelta(days=1)).timestamp())
    try:
        r = requests.get(url, params={
            "pair": symbol, "interval": 1440, "since": since
        }, timeout=20)
        resp = r.json()
        if resp.get("error"):
            return {"error": str(resp["error"])}
        result   = resp["result"]
        pair_key = [k for k in result if k != "last"][0]
        for k in result[pair_key]:
            if int(k[0]) == int(date_dt.timestamp()):
                return {
                    "kraken_open":  float(k[1]), "kraken_high": float(k[2]),
                    "kraken_low":   float(k[3]), "kraken_close":float(k[4]),
                }
    except Exception as e:
        return {"error": str(e)}
    return None


def get_coinbase_ohlcv(pair, date_str):
    """Fetch Coinbase daily OHLCV for a specific date."""
    if pair not in COINBASE_SYMBOLS:
        return None
    symbol  = COINBASE_SYMBOLS[pair]
    url     = f"https://api.exchange.coinbase.com/products/{symbol}/candles"
    date_dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    try:
        r = requests.get(url, params={
            "granularity": 86400,
            "start": date_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end":   (date_dt + timedelta(days=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }, timeout=20)
        data = r.json()
        if not data or isinstance(data, dict):
            return None
        for k in data:
            if int(k[0]) == int(date_dt.timestamp()):
                return {
                    "coinbase_open":  float(k[3]), "coinbase_high": float(k[2]),
                    "coinbase_low":   float(k[1]), "coinbase_close":float(k[4]),
                }
    except Exception as e:
        return {"error": str(e)}
    return None


print("✅ All fetch functions ready.")

✅ All fetch functions ready.


## 3. Run Diagnostic on All Suspicious Rows

In [3]:
results = []

for pair, date_str, rep_high, rep_low in SUSPICIOUS:
    print(f"\n{'─'*60}")
    print(f"  {pair}  {date_str}  "
          f"reported High={rep_high}  Low={rep_low}")
    print(f"{'─'*60}")

    row = {
        "Pair":          pair,
        "Date":          date_str,
        "Rep_High":      rep_high,
        "Rep_Low":       rep_low,
    }

    print("  [1] Binance.US trade file (raw executions)...")
    trade_result = get_binance_trade_range(pair, date_str)
    if trade_result is None:
        print("      ⚠️  Trade file not available (404)")
        row["Trade_High"] = None
        row["Trade_Low"]  = None
        row["Trade_N"]    = None
    elif "error" in trade_result:
        print(f"      ❌ Error: {trade_result['error']}")
        row["Trade_High"] = None
        row["Trade_Low"]  = None
        row["Trade_N"]    = None
    else:
        print(f"      Trades: {trade_result['n_trades']:,}")
        print(f"      Real High  : {trade_result['trade_high']:.6f}")
        print(f"      Real Low   : {trade_result['trade_low']:.6f}")
        print(f"      Real Open  : {trade_result['trade_open']:.6f}")
        print(f"      Real Close : {trade_result['trade_close']:.6f}")
        row["Trade_High"] = trade_result["trade_high"]
        row["Trade_Low"]  = trade_result["trade_low"]
        row["Trade_N"]    = trade_result["n_trades"]

    print("  [2] Binance.US kline re-fetch...")
    kline = get_binance_kline(pair, date_str)
    if kline is None:
        print("      ⚠️  Kline file not available")
        row["Kline_High"] = None
        row["Kline_Low"]  = None
    elif "error" in kline:
        print(f"      ❌ Error: {kline['error']}")
        row["Kline_High"] = None
        row["Kline_Low"]  = None
    else:
        print(f"      Kline High  : {kline['kline_high']:.6f}")
        print(f"      Kline Low   : {kline['kline_low']:.6f}")
        row["Kline_High"] = kline["kline_high"]
        row["Kline_Low"]  = kline["kline_low"]

    print("  [3] Kraken cross-check...")
    kraken = get_kraken_ohlcv(pair, date_str)
    if kraken is None:
        print("      ⚠️  No data")
        row["Kraken_High"] = None
        row["Kraken_Low"]  = None
    elif "error" in kraken:
        print(f"      ❌ Error: {kraken['error']}")
        row["Kraken_High"] = None
        row["Kraken_Low"]  = None
    else:
        print(f"      Kraken High  : {kraken['kraken_high']:.6f}")
        print(f"      Kraken Low   : {kraken['kraken_low']:.6f}")
        row["Kraken_High"] = kraken["kraken_high"]
        row["Kraken_Low"]  = kraken["kraken_low"]

    if pair in COINBASE_SYMBOLS:
        print("  [4] Coinbase cross-check...")
        cb = get_coinbase_ohlcv(pair, date_str)
        if cb is None:
            print("      ⚠️  No data (pair may not have been listed yet)")
            row["Coinbase_High"] = None
            row["Coinbase_Low"]  = None
        elif "error" in cb:
            print(f"      ❌ Error: {cb['error']}")
            row["Coinbase_High"] = None
            row["Coinbase_Low"]  = None
        else:
            print(f"      Coinbase High  : {cb['coinbase_high']:.6f}")
            print(f"      Coinbase Low   : {cb['coinbase_low']:.6f}")
            row["Coinbase_High"] = cb["coinbase_high"]
            row["Coinbase_Low"]  = cb["coinbase_low"]
    else:
        row["Coinbase_High"] = None
        row["Coinbase_Low"]  = None

    best_high = None
    best_low  = None
    best_src  = None

    candidates = [
        ("trade",    row.get("Trade_High"),  row.get("Trade_Low")),
        ("kline",    row.get("Kline_High"),  row.get("Kline_Low")),
        ("kraken",   row.get("Kraken_High"), row.get("Kraken_Low")),
        ("coinbase", row.get("Coinbase_High"),row.get("Coinbase_Low")),
    ]
    for src_name, h, l in candidates:
        if h is not None and l is not None:
            if h <= PEG_HIGH and l >= PEG_LOW:
                best_high = h
                best_low  = l
                best_src  = src_name
                break

    row["Best_High"]   = best_high
    row["Best_Low"]    = best_low
    row["Best_Source"] = best_src

    if best_src:
        print(f"\n  ✅ VERDICT: Use {best_src} values — "
              f"High={best_high:.6f}  Low={best_low:.6f}")
    else:
        print(f"\n  ⚠️  VERDICT: All sources outside peg band or unavailable — "
              f"manual review required")

    results.append(row)
    time.sleep(0.3)

results_df = pd.DataFrame(results)
print(f"\n\n✅ Diagnostic complete — {len(results_df)} rows investigated")


────────────────────────────────────────────────────────────
  USDT/USD  2020-02-19  reported High=5.0  Low=1.0
────────────────────────────────────────────────────────────
  [1] Binance.US trade file (raw executions)...
      Trades: 13,937
      Real High  : 4.999900
      Real Low   : 0.997700
      Real Open  : 1.001900
      Real Close : 1.000000
  [2] Binance.US kline re-fetch...
      Kline High  : 4.999900
      Kline Low   : 0.997700
  [3] Kraken cross-check...
      ⚠️  No data
  [4] Coinbase cross-check...
      ⚠️  No data (pair may not have been listed yet)

  ⚠️  VERDICT: All sources outside peg band or unavailable — manual review required

────────────────────────────────────────────────────────────
  USDT/USD  2020-03-13  reported High=4.0  Low=1.0
────────────────────────────────────────────────────────────
  [1] Binance.US trade file (raw executions)...
      Trades: 16,518
      Real High  : 4.000000
      Real Low   : 0.996100
      Real Open  : 1.001700
      Real

## 4. Summary Table

In [4]:
def fmt(v):
    return f"{v:.4f}" if v is not None else "N/A"

print("═" * 110)
print("  DIAGNOSTIC SUMMARY")
print("═" * 110)
print(f"  {'Pair':<12} {'Date':<12} "
      f"{'Rep H':>7} {'Rep L':>7} "
      f"{'Trade H':>8} {'Trade L':>8} "
      f"{'Kraken H':>9} {'Kraken L':>9} "
      f"{'CB H':>7} {'CB L':>7} "
      f"{'Best H':>7} {'Best L':>7}  Best src")
print("─" * 110)

for _, r in results_df.iterrows():
    flag = " ←BAD" if r["Rep_High"] > PEG_HIGH or r["Rep_Low"] < PEG_LOW else ""
    print(f"  {r['Pair']:<12} {r['Date']:<12} "
          f"{fmt(r['Rep_High']):>7} {fmt(r['Rep_Low']):>7} "
          f"{fmt(r.get('Trade_High')):>8} {fmt(r.get('Trade_Low')):>8} "
          f"{fmt(r.get('Kraken_High')):>9} {fmt(r.get('Kraken_Low')):>9} "
          f"{fmt(r.get('Coinbase_High')):>7} {fmt(r.get('Coinbase_Low')):>7} "
          f"{fmt(r.get('Best_High')):>7} {fmt(r.get('Best_Low')):>7}  "
          f"{r.get('Best_Source') or 'MANUAL REVIEW'}{flag}")

print("═" * 110)

fixable = results_df[results_df["Best_Source"].notna()]
manual  = results_df[results_df["Best_Source"].isna()]
print(f"\n  Fixable (replacement source found) : {len(fixable)}")
print(f"  Needs manual review                : {len(manual)}")

if not manual.empty:
    print("\n  Rows needing manual review:")
    for _, r in manual.iterrows():
        print(f"    {r['Pair']}  {r['Date']}  "
              f"Rep High={r['Rep_High']}  Rep Low={r['Rep_Low']}")

display(results_df)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC SUMMARY
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Pair         Date           Rep H   Rep L  Trade H  Trade L  Kraken H  Kraken L    CB H    CB L  Best H  Best L  Best src
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  USDT/USD     2020-02-19    5.0000  1.0000   4.9999   0.9977       N/A       N/A     nan     nan     nan     nan  MANUAL REVIEW ←BAD
  USDT/USD     2020-03-13    4.0000  1.0000   4.0000   0.9961       N/A       N/A     nan     nan     nan     nan  MANUAL REVIEW ←BAD
  USDT/USD     2023-05-05    3.4800  1.0000   3.4799   1.0018       N/A       N/A  1.0011  1.0002  1.0011  1.0002  coinbase ←BAD
  USDT/USD     2023-06-06    4.5000  1.0000   4.5000   0.9999       N/A       N/A  1.0004  0.9990  1.0004  0.9990  coinbase ←

,Pair,Date,Rep_High,Rep_Low,Trade_High,Trade_Low,Trade_N,Kline_High,Kline_Low,Kraken_High,Kraken_Low,Coinbase_High,Coinbase_Low,Best_High,Best_Low,Best_Source
0,USDT/USD,2020-02-19,5.00,1.00,4.9999,0.9977,13937,4.9999,0.9977,None,None,NaN,NaN,NaN,NaN,None
1,USDT/USD,2020-03-13,4.00,1.00,4.0000,0.9961,16518,4.0000,0.9961,None,None,NaN,NaN,NaN,NaN,None
2,USDT/USD,2023-05-05,3.48,1.00,3.4799,1.0018,23233,3.4799,1.0018,None,None,1.00109,1.00024,1.00109,1.00024,coinbase
3,USDT/USD,2023-06-06,4.50,1.00,4.5000,0.9999,22484,4.5000,0.9999,None,None,1.00040,0.99900,1.00040,0.99900,coinbase
4,USDT/USD,2023-06-07,1.70,1.00,1.6996,1.0030,46427,1.6996,1.0030,None,None,1.00040,0.99996,1.00040,0.99996,coinbase
5,USDT/USD,2023-07-14,0.92,0.70,0.9185,0.7044,17520,0.9185,0.7044,None,None,1.00070,0.99961,1.00070,0.99961,coinbase
6,USDC/USD,2021-01-12,1.50,1.00,5.0000,0.9999,375,5.0000,0.9999,None,None,NaN,NaN,NaN,NaN,None
7,USDC/USD,2021-01-13,2.00,1.00,2.0000,0.9990,400,2.0000,0.9990,None,None,NaN,NaN,NaN,NaN,None
8,USDC/USD,2021-05-19,1.35,0.98,1.3490,0.9800,6008,1.3490,0.9800,None,None,NaN,NaN,NaN,NaN,None
9,USDC/USD,2022-12-13,4.00,1.00,3.9999,1.0001,4493,3.9999,1.0001,None,None,NaN,NaN,NaN,NaN,None


## 5. Export Diagnostic Results

In [5]:
results_df.to_csv(DIAGNOSTIC_OUTPUT_CSV, index=False)
print("✅ Saved: stablecoin_diagnostic.csv")
print()
print("Next step: once you review this output, run the fix cell below")
print("to apply the Best_High / Best_Low values back into ohlcv_final.xlsx")

✅ Saved: stablecoin_diagnostic.csv

Next step: once you review this output, run the fix cell below
to apply the Best_High / Best_Low values back into ohlcv_final.xlsx


## 6. Apply Fixes to ohlcv_final.xlsx

Two correction strategies are applied automatically based on what the diagnostic found:

**Strategy A — Cross-exchange replacement** (`Best_Source` = coinbase/kraken)  
Used when an independent exchange had a clean value within the peg band.  
Replaces High and Low with the cross-exchange values.  
Excel highlight: **pale blue**

**Strategy B — Close-anchor correction** (`Best_Source` = NaN, manual review group)  
Used when all sources confirmed the anomalous trade happened (trade file agreed) but no
independent exchange exists to provide a reference price. This is the thin-market case —
a single stale order executing on a near-empty book in 2020–2021.  
Sets `High = Close` and `Low = min(reported_Low, Close)`, preserving the close price
while removing the uninformative intraday spike.  
Excel highlight: **pale orange**

Run this cell only after reviewing the diagnostic table above.

In [6]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment

EXCEL_FILE = INPUT_EXCEL_FILE
FILL_CROSS_EXCH  = PatternFill("solid", start_color="D9E1F2")
FILL_CLOSE_ANCHOR = PatternFill("solid", start_color="FCE4D6")
thin   = Border(left=Side(style="thin"),  right=Side(style="thin"),
                top=Side(style="thin"),   bottom=Side(style="thin"))
center = Alignment(horizontal="center")


def _style_cell(cell, fill):
    cell.fill            = fill
    cell.font            = Font(name="Arial", size=10, bold=True)
    cell.border          = thin
    cell.alignment       = center
    cell.number_format   = "#,##0.00000000"


def _find_target_row(ws, date_str, date_col):
    for r in range(2, ws.max_row + 1):
        if str(ws.cell(row=r, column=date_col).value)[:10] == date_str:
            return r
    return None


wb          = load_workbook(EXCEL_FILE)
fixed_a     = 0
fixed_b     = 0
skipped     = 0

print("═" * 70)
print("  APPLYING FIXES")
print("═" * 70)

for _, diag in results_df.iterrows():
    pair       = diag["Pair"]
    date_str   = diag["Date"]
    rep_high   = diag["Rep_High"]
    rep_low    = diag["Rep_Low"]
    best_src   = diag.get("Best_Source")

    sheet_name = pair.replace("/", "_")
    if sheet_name not in wb.sheetnames:
        print(f"  ⚠️  Sheet '{sheet_name}' not found — skipped")
        skipped += 1
        continue

    ws      = wb[sheet_name]
    col_map = {ws.cell(row=1, column=c).value: c
               for c in range(1, ws.max_column + 1)}

    target_row = _find_target_row(ws, date_str, col_map["Date"])
    if target_row is None:
        print(f"  ⚠️  {pair} {date_str} — date not found in sheet, skipped")
        skipped += 1
        continue

    current_close = ws.cell(row=target_row, column=col_map["Close"]).value

    if pd.notna(best_src):
        new_high  = diag["Best_High"]
        new_low   = diag["Best_Low"]
        fill      = FILL_CROSS_EXCH
        src_note  = f"H/L replaced from {best_src}"
        strategy  = "A (cross-exchange)"
        fixed_a  += 1
    else:
        new_high  = float(current_close)
        new_low   = min(float(rep_low), float(current_close))
        fill      = FILL_CLOSE_ANCHOR
        src_note  = "H/L close-anchored (thin-market stale-order spike removed)"
        strategy  = "B (close-anchor)"
        fixed_b  += 1

    for col_name, new_val in [("High", new_high), ("Low", new_low)]:
        c_idx = col_map.get(col_name)
        if c_idx:
            _style_cell(
                ws.cell(row=target_row, column=c_idx, value=new_val),
                fill
            )

    src_col = col_map.get("Source")
    if src_col:
        orig = ws.cell(row=target_row, column=src_col).value or ""
        ws.cell(row=target_row, column=src_col).value = f"{orig} [{src_note}]"

    print(f"  ✅ {strategy:<22}  {pair}  {date_str}  "
          f"High: {rep_high:.4f} → {new_high:.6f}  "
          f"Low: {rep_low:.4f} → {new_low:.6f}")

wb.save(EXCEL_FILE)

print(f"\n{'═'*70}")
print(f"  Strategy A (cross-exchange replacement) : {fixed_a} rows")
print(f"  Strategy B (close-anchor correction)    : {fixed_b} rows")
print(f"  Skipped                                 : {skipped} rows")
print(f"  Total rows touched                      : {fixed_a + fixed_b}")
print(f"\n  Saved to {EXCEL_FILE}")
print(f"  Pale blue  = Strategy A (cross-exchange replacement)")
print(f"  Pale orange = Strategy B (close-anchor, thin-market correction)")
print(f"  Source column updated on all corrected rows.")

══════════════════════════════════════════════════════════════════════
  APPLYING FIXES
══════════════════════════════════════════════════════════════════════
  ✅ B (close-anchor)        USDT/USD  2020-02-19  High: 5.0000 → 1.000000  Low: 1.0000 → 1.000000
  ✅ B (close-anchor)        USDT/USD  2020-03-13  High: 4.0000 → 1.008200  Low: 1.0000 → 1.000000
  ✅ A (cross-exchange)      USDT/USD  2023-05-05  High: 3.4800 → 1.001090  Low: 1.0000 → 1.000240
  ✅ A (cross-exchange)      USDT/USD  2023-06-06  High: 4.5000 → 1.000400  Low: 1.0000 → 0.999000
  ✅ A (cross-exchange)      USDT/USD  2023-06-07  High: 1.7000 → 1.000400  Low: 1.0000 → 0.999960
  ✅ A (cross-exchange)      USDT/USD  2023-07-14  High: 0.9200 → 1.000700  Low: 0.7000 → 0.999610
  ✅ B (close-anchor)        USDC/USD  2021-01-12  High: 1.5000 → 1.000000  Low: 1.0000 → 1.000000
  ✅ B (close-anchor)        USDC/USD  2021-01-13  High: 2.0000 → 0.999900  Low: 1.0000 → 0.999900
  ✅ B (close-anchor)        USDC/USD  2021-05-19  High: 1